# Section 5: Monitoring & Observability
## AI-Native Software Architecture — O'Reilly Course

Traditional monitoring tells us whether the service is up.
LLM observability tells us whether the system behaved correctly.

We will add:
- tracing
- prompt/version tracking
- token/cost approximations
- deterministic evaluation
- LLM-as-a-judge rubric

In [28]:
import support_utils
from support_utils import (
    call_llm,
    primary_issue,
    structured_support_prompt,
    escalation_response, blocked_response,
    FORBIDDEN_CLAIMS, parse_json_response,
)
from datetime import datetime, UTC
from typing import Any, Dict, List, Optional
import json

In [29]:
# Toggle LLM mode without editing support_utils.py or restarting the kernel.
# Gemini (Vertex AI) is used when gemini_client is initialised in support_utils.
import support_utils
support_utils.USE_REAL_LLM = False   # set True to call a real LLM
print(f"USE_REAL_LLM = {support_utils.USE_REAL_LLM}")

USE_REAL_LLM = False


## Step 5.1: Trace Events

A trace captures what happened during a request.
In production this could go to LangSmith, Langfuse, Arize, Datadog, or Braintrust.
For this notebook we use an in-memory list.

In [30]:
TRACE_LOGS: List[Dict[str, Any]] = []

def estimate_tokens(text: str) -> int:
    return max(1, len(text) // 4)

def log_trace(request_id: str, stage: str, event: str, payload: Dict[str, Any]) -> None:
    TRACE_LOGS.append({
        "timestamp": datetime.now(UTC).isoformat(),
        "request_id": request_id,
        "stage": stage,
        "event": event,
        "payload": payload,
    })

def view_traces(request_id: Optional[str] = None) -> List[Dict[str, Any]]:
    rows = TRACE_LOGS
    if request_id:
        rows = [r for r in rows if r["request_id"] == request_id]
    return rows

def print_traces(request_id: Optional[str] = None) -> None:
    for r in view_traces(request_id):
        print(f"\n🧾 [{r['timestamp']}]")
        print(f"Request: {r['request_id']}")
        print(f"Stage: {r['stage']} | Event: {r['event']}")
        print(f"Payload: {r['payload']}")

log_trace("demo_1", "setup", "notebook_trace_initialized", {"status": "ok"})
print_traces()


🧾 [2026-05-07T16:56:40.430050+00:00]
Request: demo_1
Stage: setup | Event: notebook_trace_initialized
Payload: {'status': 'ok'}


In [ ]:
## Generate a real support response and trace it
# Auto-increment request ID each run so traces accumulate
if "_req_counter" not in dir():
    _req_counter = 0
_req_counter += 1
request_id = f"req_trace_{_req_counter:02d}"

prompt = structured_support_prompt(primary_issue)

log_trace(request_id, "input", "issue_received", {"issue": primary_issue})
log_trace(request_id, "prompt", "prompt_built", {"prompt_preview": prompt[:200]})

llm_output_raw = call_llm(prompt, force_json=True)
log_trace(request_id, "llm", "response_received", {
    "output_preview": llm_output_raw[:300],
    "tokens_in": estimate_tokens(prompt),
    "tokens_out": estimate_tokens(llm_output_raw),
})

llm_output, parse_err = parse_json_response(llm_output_raw)
if parse_err or not isinstance(llm_output, dict):
    print(f"Parse error: {parse_err or type(llm_output)}. Using empty dict.")
    llm_output = {}

print(f"Request ID: {request_id}")
print(f"Total traces so far: {len(TRACE_LOGS)}")
print("\nLLM response:")
print(json.dumps(llm_output, indent=2))
print("\nTraces for this request:")
print_traces(request_id)

## Step 5.2: Evaluation Metrics

We evaluate outputs using deterministic checks.

In production you may combine:
- deterministic validation
- reference-based checks
- LLM-as-a-judge
- human review
- product/business signals

In [ ]:
def evaluate_response(output: Dict[str, Any]) -> Dict[str, Any]:
    message = json.dumps(output).lower()
    return {
        "schema_present": isinstance(output, dict),
        "no_unsafe_refund_claim": not any(claim in message for claim in FORBIDDEN_CLAIMS),
        "has_next_action": bool(output.get("next_action") or output.get("message")),
        "requires_human_for_refund": (
            "refund" not in message
            or "human" in message
            or "escalat" in message
            or "verify" in message
        ),
    }

def score_eval(eval_result: Dict[str, bool]) -> float:
    values = list(eval_result.values())
    return sum(bool(v) for v in values) / len(values)

sample_eval = evaluate_response(escalation_response(primary_issue, "Refund requires human approval"))
print(sample_eval)
print("Score:", score_eval(sample_eval))

## Step 5.3: LLM-as-a-Judge Rubric

This rubric mirrors the slide exercise on evaluation dimensions:
- Faithfulness
- Conciseness
- Schema adherence

In [ ]:
JUDGE_RUBRIC = """
You are evaluating the output of a support assistant.

Score the response from 1-5 on:

1. Faithfulness:
Does the response stay grounded in the provided policy and avoid inventing actions?

2. Conciseness:
Is the response clear and direct without unnecessary detail?

3. Schema adherence:
Does the response follow the expected JSON or response structure?

Return JSON:
{
  "faithfulness": number,
  "conciseness": number,
  "schema_adherence": number,
  "reasoning": "short explanation"
}
"""

def llm_judge_prompt(output: Dict[str, Any], context: str) -> str:
    return f"""
{JUDGE_RUBRIC}

Context:
{context}

Assistant output:
{json.dumps(output, indent=2)}
"""

print(llm_judge_prompt(
    escalation_response(primary_issue, "Refund requires human approval"),
    "Duplicate charge refunds require human approval.",
))


In [ ]:
## Call the LLM judge on the most recent response
context = "Duplicate charge refunds require human verification before approval."
judge_prompt = llm_judge_prompt(llm_output, context)

log_trace(request_id, "evaluation", "judge_called", {"context": context})
judge_raw = call_llm(judge_prompt, temperature=0.2, force_json=True)
judge_result, judge_err = parse_json_response(judge_raw)
if judge_err or not isinstance(judge_result, dict):
    print(f"Judge parse error: {judge_err or type(judge_result)}. Raw:\n{judge_raw}")
    judge_result = {}
log_trace(request_id, "evaluation", "judge_result", judge_result)

print(f"Judge scores for {request_id}:")
print(json.dumps(judge_result, indent=2))

# Combine deterministic + LLM-judge scores
det_eval = evaluate_response(llm_output)
det_score = score_eval(det_eval)
print(f"\nDeterministic score: {det_score:.2f}")
if isinstance(judge_result, dict) and "faithfulness" in judge_result:
    llm_avg = sum([judge_result.get("faithfulness", 0),
                   judge_result.get("conciseness", 0),
                   judge_result.get("schema_adherence", 0)]) / 15
    print(f"LLM judge score (normalised 0-1): {llm_avg:.2f}")

print(f"\nFull trace for {request_id}:")
print_traces(request_id)

print(f"\n--- All request IDs traced so far: {sorted(set(r['request_id'] for r in TRACE_LOGS))} ---")

## Section 5 takeaway

You cannot scale what you cannot measure.

Observability gives teams a way to:
- debug failures
- compare prompt versions
- catch regressions
- understand cost and latency
- improve the system over time

**Next:** Section 6 — composing everything into an end-to-end pipeline.